In [101]:
import cosmic.utils as cu
import astropy.io.fits as fits
from astropy.table import Table
import numpy as np
import pandas as pd
from cosmic.panstarrs_dr2.dataProcess import correct_extinction

# 严格清理

In [102]:
df0_raw = read_fits_to_df('./ps1dr2_NonGal.0.fits')
df1_raw = read_fits_to_df('./ps1dr2_NonGal.1.fits')

In [ ]:
def read_fits_to_df(path):
    """读取 FITS 文件并转换为 pandas DataFrame"""
    with fits.open(path) as hdul:
        tab = Table(hdul[1].data)

    # 展平多维列
    for col in tab.colnames:
        if len(tab[col].shape) > 1:
            tab[col] = tab[col][:, 0]

    return tab.to_pandas()

def clean_panstarrs(df):
    """PanSTARRS 数据清理 - 排除星系，保留高质量点源"""

    print(f'原始数据: {len(df)} 行')

    # 0. 先展平所有多维列
    for col in df.columns:
        if len(df[col].shape) > 1:
            df[col] = df[col][:, 0]
    
    # 清理缺失值
    df = df.replace([-99.0, -99, -999.0, -9999.0, -999, -9999], np.nan)
    df = df.dropna()
    print('After dropna: ', len(df))

    # 2. nDetections >= 1
    df = df[np.array(df['nDetections']).flatten() >= 1]
    print(f'nDetections 过滤后: {len(df)} 行')

    # 3. objInfoFlag - 排除星系和问题源
    EXT_FLAGS = 0x00000001 | 0x00000002
    BAD_OBJ = 0x00000020 | 0x00000040 | 0x00080000 | 0x00100000
    flags = np.nan_to_num(np.array(df['objInfoFlag']).flatten().astype(np.int64), nan=0)
    mask = ((flags & EXT_FLAGS) == 0) & ((flags & BAD_OBJ) == 0)
    df = df[mask]
    print(f'objInfoFlag 过滤后: {len(df)} 行')

    # 4. qualityFlag - 保留 PRIMARY
    flags_to_exclude = [0x00000040, 0x00000080]
    df = df[~df['qualityFlag'].isin(flags_to_exclude)]
    print(f'qualityFlag 过滤后: {len(df)} 行')

    # 只排除最严重的问题
    SUSPICIOUS_MASK = (
        0x8 | 0x400 | 0x800 | 0x1000 | 
        0x2000 | 0x10000 | 0x400000 | 0x1000000
    )
    mask = np.ones(len(df), dtype=bool)
    for b in ['g', 'r', 'i', 'z', 'y']:
        col = f'{b}infoFlag'
        if col in df.columns:
            flags = np.nan_to_num(np.array(df[col]).flatten().astype(np.int64), nan=0)
            mask &= (flags & SUSPICIOUS_MASK) == 0
    df = df[mask]
    print(f'infoFlag 过滤后: {len(df)} 行')
    
    # PSF - Kron 星等差
    threshold = 0.1
    cond_g = abs(df['gPSFMag'] - df['gKronMag']) < threshold
    cond_r = abs(df['rPSFMag'] - df['rKronMag']) < threshold # 本身r波段就 < 0.01
    cond_i = abs(df['iPSFMag'] - df['iKronMag']) < threshold
    df = df[cond_r & cond_i & cond_g]
    print(f'PSF - Kron 星等差过滤后: {len(df)} 行')

    return df


df0 = df0_raw.copy()
df1 = df1_raw.copy()

df0 = clean_panstarrs(df0)
df1 = clean_panstarrs(df1)
# dered
df0 = correct_extinction(df0)
df1 = correct_extinction(df1)
# concat
df = pd.concat([df0, df1], ignore_index=True)

原始数据: 5000000 行
After dropna:  5000000
nDetections 过滤后: 4997488 行
objInfoFlag 过滤后: 4993658 行
qualityFlag 过滤后: 4993658 行
infoFlag 过滤后: 3179679 行
PSF - Kron 星等差过滤后: 1379115 行
原始数据: 5000000 行
After dropna:  5000000
nDetections 过滤后: 4998616 行
objInfoFlag 过滤后: 4995696 行
qualityFlag 过滤后: 4995696 行
infoFlag 过滤后: 4245568 行
PSF - Kron 星等差过滤后: 1943008 行


In [106]:
df.shape

(3322123, 68)

In [107]:
cols = [
    'objID', 'raStack', 'decStack',
    'gKronMag_dered', 'rKronMag_dered', 'iKronMag_dered', 'zKronMag_dered', 'yKronMag_dered',
    'gPSFMag_dered', 'rPSFMag_dered', 'iPSFMag_dered', 'zPSFMag_dered', 'yPSFMag_dered', 
    'gApMag_dered', 'rApMag_dered', 'iApMag_dered', 'zApMag_dered', 'yApMag_dered',
    'gKronMagErr', 'rKronMagErr', 'iKronMagErr', 'zKronMagErr', 'yKronMagErr',
    'gPSFMagErr', 'rPSFMagErr', 'iPSFMagErr', 'zPSFMagErr', 'yPSFMagErr',
    'gApMagErr', 'rApMagErr', 'iApMagErr', 'zApMagErr', 'yApMagErr',
]
df['label'] = 0
rename = {'raStack': 'ra', 'decStack': 'dec'}
df = df[cols].rename(columns=rename)

output_path = './PS1DR2_NonGal_clean_gri.fits'
cu.savefile(df, output_path)

In [109]:
df.columns

Index(['objID', 'ra', 'dec', 'gKronMag_dered', 'rKronMag_dered',
       'iKronMag_dered', 'zKronMag_dered', 'yKronMag_dered', 'gPSFMag_dered',
       'rPSFMag_dered', 'iPSFMag_dered', 'zPSFMag_dered', 'yPSFMag_dered',
       'gApMag_dered', 'rApMag_dered', 'iApMag_dered', 'zApMag_dered',
       'yApMag_dered', 'gKronMagErr', 'rKronMagErr', 'iKronMagErr',
       'zKronMagErr', 'yKronMagErr', 'gPSFMagErr', 'rPSFMagErr', 'iPSFMagErr',
       'zPSFMagErr', 'yPSFMagErr', 'gApMagErr', 'rApMagErr', 'iApMagErr',
       'zApMagErr', 'yApMagErr'],
      dtype='object')

# 宽松清理

In [110]:
df0_raw = read_fits_to_df('./ps1dr2_NonGal.0.fits')
df1_raw = read_fits_to_df('./ps1dr2_NonGal.1.fits')

In [111]:
def read_fits_to_df(path):
    """读取 FITS 文件并转换为 pandas DataFrame"""
    with fits.open(path) as hdul:
        tab = Table(hdul[1].data)

    # 展平多维列
    for col in tab.colnames:
        if len(tab[col].shape) > 1:
            tab[col] = tab[col][:, 0]

    return tab.to_pandas()

def clean_panstarrs(df):
    """PanSTARRS 数据清理 - 排除星系，保留高质量点源"""

    print(f'原始数据: {len(df)} 行')

    # 0. 先展平所有多维列
    for col in df.columns:
        if len(df[col].shape) > 1:
            df[col] = df[col][:, 0]
    
    # 清理缺失值
    df = df.replace([-99.0, -99, -999.0, -9999.0, -999, -9999], np.nan)
    df = df.dropna()
    print('After dropna: ', len(df))

    # 2. nDetections >= 1
    df = df[np.array(df['nDetections']).flatten() >= 1]
    print(f'nDetections 过滤后: {len(df)} 行')

    # 3. objInfoFlag - 排除星系和问题源
    EXT_FLAGS = 0x00000001 | 0x00000002
    BAD_OBJ = 0x00000020 | 0x00000040 | 0x00080000 | 0x00100000
    flags = np.nan_to_num(np.array(df['objInfoFlag']).flatten().astype(np.int64), nan=0)
    mask = ((flags & EXT_FLAGS) == 0) & ((flags & BAD_OBJ) == 0)
    df = df[mask]
    print(f'objInfoFlag 过滤后: {len(df)} 行')

    # 4. qualityFlag - 保留 PRIMARY
    flags_to_exclude = [0x00000040, 0x00000080]
    df = df[~df['qualityFlag'].isin(flags_to_exclude)]
    print(f'qualityFlag 过滤后: {len(df)} 行')

    # 只排除最严重的问题
    SUSPICIOUS_MASK = (
        0x8 | 0x400 | 0x800 | 0x1000 | 
        0x2000 | 0x10000 | 0x400000 | 0x1000000
    )
    mask = np.ones(len(df), dtype=bool)
    for b in ['g', 'r', 'i', 'z', 'y']:
        col = f'{b}infoFlag'
        if col in df.columns:
            flags = np.nan_to_num(np.array(df[col]).flatten().astype(np.int64), nan=0)
            mask &= (flags & SUSPICIOUS_MASK) == 0
    df = df[mask]
    print(f'infoFlag 过滤后: {len(df)} 行')
    
    # PSF - Kron 星等差
    threshold = 0.01
    cond_r = abs(df['rPSFMag'] - df['rKronMag']) < threshold  # 本身r波段就 < 0.01
    df = df[cond_r]
    print(f'PSF - Kron 星等差过滤后: {len(df)} 行')

    return df


df0 = df0_raw.copy()
df1 = df1_raw.copy()

df0 = clean_panstarrs(df0)
df1 = clean_panstarrs(df1)
# dered
df0 = correct_extinction(df0)
df1 = correct_extinction(df1)
# concat
df = pd.concat([df0, df1], ignore_index=True)

原始数据: 5000000 行
After dropna:  5000000
nDetections 过滤后: 4997488 行
objInfoFlag 过滤后: 4993658 行
qualityFlag 过滤后: 4993658 行
infoFlag 过滤后: 3179679 行
PSF - Kron 星等差过滤后: 3179679 行
原始数据: 5000000 行
After dropna:  5000000
nDetections 过滤后: 4998616 行
objInfoFlag 过滤后: 4995696 行
qualityFlag 过滤后: 4995696 行
infoFlag 过滤后: 4245568 行
PSF - Kron 星等差过滤后: 4245568 行


In [112]:
cols = [
    'objID', 'raStack', 'decStack',
    'gKronMag_dered', 'rKronMag_dered', 'iKronMag_dered', 'zKronMag_dered', 'yKronMag_dered',
    'gPSFMag_dered', 'rPSFMag_dered', 'iPSFMag_dered', 'zPSFMag_dered', 'yPSFMag_dered', 
    'gApMag_dered', 'rApMag_dered', 'iApMag_dered', 'zApMag_dered', 'yApMag_dered',
    'gKronMagErr', 'rKronMagErr', 'iKronMagErr', 'zKronMagErr', 'yKronMagErr',
    'gPSFMagErr', 'rPSFMagErr', 'iPSFMagErr', 'zPSFMagErr', 'yPSFMagErr',
    'gApMagErr', 'rApMagErr', 'iApMagErr', 'zApMagErr', 'yApMagErr',
]
df['label'] = 0
rename = {'raStack': 'ra', 'decStack': 'dec'}
df = df[cols].rename(columns=rename)

output_path = './PS1DR2_NonGal_clean_r.fits'
cu.savefile(df, output_path)